# 🛰️ OrbitGPT — train a tiny language model in Colab

This notebook trains a small GPT **from scratch** and lets you chat with it. No API keys, no downloads except the training text, PyTorch only.

1. **Runtime → Change runtime type → T4 GPU** (strongly recommended; CPU works but is ~10× slower)
2. Run the cells in order
3. Training takes ~2 minutes on a T4; chatting starts automatically when it's done

Everything lives in one self-contained file (`orbit_gpt_colab.py`) with a `CONFIG` block at the top you can edit.

In [ ]:
# 1) Check the machine -----------------------------------------------------------
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'no GPU: CPU runtime (slower)'
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2) Get the code (one file, ~90 KB) ---------------------------------------------
import os, urllib.request

URLS = [
    'https://raw.githubusercontent.com/Orbitlol/orbit-gpt/main/colab/orbit_gpt_colab.py',
    'https://raw.githubusercontent.com/Orbitlol/orbit-gpt/arena/01a08c69-orbit-gpt/colab/orbit_gpt_colab.py',
]
if not (os.path.exists('orbit_gpt_colab.py') and os.path.getsize('orbit_gpt_colab.py') > 5000):
    for url in URLS:
        try:
            urllib.request.urlretrieve(url, 'orbit_gpt_colab.py')
            print('downloaded from', url)
            break
        except Exception as e:
            print('failed:', url, '->', e)
print('orbit_gpt_colab.py:', os.path.getsize('orbit_gpt_colab.py'), 'bytes')

In [ ]:
# 3) Train ------------------------------------------------------------------------
#    Edit options here, e.g. --preset mini --corpus ./my_notes.txt --max-steps 4000
%run orbit_gpt_colab.py

## Chat with it again (no retraining)

The cell above already starts a chat loop. If you interrupted it, or restarted the runtime after training, just reload the saved checkpoint.

In [ ]:
# 4) Reload the trained model and chat --------------------------------------------
%run orbit_gpt_colab.py --no-train --out-dir /content/orbit_model

## Train on your own text

Upload a `.txt` file (or a folder of them) and point `--corpus` at it. A few hundred KB of text is plenty for the `micro` model.

In [ ]:
# 5) Upload your own corpus, then train on it --------------------------------------
from google.colab import files
uploaded = files.upload()          # pick one or more .txt files
print(sorted(uploaded))

# then, for a single file:
# !python orbit_gpt_colab.py --corpus my_notes.txt --preset micro --max-steps 3000
# ...or mix it with the built-in assistant corpus so it can chat:
# !python orbit_gpt_colab.py --corpus my_notes.txt,orbit-chat:12 --preset micro

In [ ]:
# 6) Download your model (weights + tokenizer) -------------------------------------
import shutil, os
if os.path.isdir('/content/orbit_model'):
    path = shutil.make_archive('/content/orbit_model', 'zip', '/content/orbit_model')
    print(path)
    from google.colab import files; files.download(path)
else:
    print('nothing trained yet - run the training cell first')

## Tips

* Better text → bigger preset (`mini`), more steps (`--max-steps 5000`), more data.
* Repetitive answers → raise the temperature (in chat: `/temp 1.0`).
* Start over → `/reset` in the chat box.
* Out of memory → smaller `--batch-size` (e.g. 16).

The model is a *toy* by modern standards — expect charming, often wrong, sometimes surprising output. That's the fun part.

Repo: <https://github.com/Orbitlol/orbit-gpt>